In [69]:
from dotenv import load_dotenv
load_dotenv()
from tqdm import tqdm
from pathlib import Path

import polars as pl
from utils import get_min_max_scores
from sklearn.model_selection import train_test_split

In [70]:
DATASET_PATH = "./dataset/asap_with_traits.csv"
LLM_PROMPT_PATH = "./llm_prompts/few_shot_user.md"
ESSAY_PROMPT_DIR = "./llm_prompts/asap_original/"
ESSAY_RUBRIC_DIR = "./llm_prompts/original_trait/"
TARGET_PROMPT_ID = 1
ATT = "overall"
NUM_FEW_SHOT = 30

MODEL = "Qwen/Qwen3.5-9B"
MODEL_FAMILIY = "qwen"
OUTDIR = Path(f"out/few_shot/{MODEL}/{TARGET_PROMPT_ID}/{ATT}")
JSONL_PATH = OUTDIR / "results.jsonl"

In [71]:
asap = pl.read_csv(DATASET_PATH, infer_schema_length=20000)
asap = asap.filter(pl.col("essay_set") == TARGET_PROMPT_ID)
asap = asap.select(["essay_id", "essay", ATT])

few_shot_df, test_df = train_test_split(asap, test_size=len(asap)-NUM_FEW_SHOT, random_state=12, shuffle=True)

print(f"few_shot_df len: {len(few_shot_df)}")
print(f"test_df len: {len(test_df)}")

few_shot_df len: 30
test_df len: 1753


In [72]:
few_shot_df

essay_id,essay,overall
i64,str,i64
1381,"""Dear Local Newspaper, I think …",8
450,"""Dear Readers, I believe that s…",7
1147,"""Dear @ORGANIZATION1 staff, Are…",9
1699,"""Dear Newspaper, I think that c…",8
1446,"""Dear local newspaper, @CAPS1 p…",10
…,…,…
1671,"""@CAPS8 more and more technolog…",11
1270,"""Dear Local Newspaper, I believ…",10
1282,"""Dear newspaper editor, @PERCEN…",10


In [73]:
# Create LLM queries
minscore, maxscore = get_min_max_scores(TARGET_PROMPT_ID, ATT)
queries = {}


# Create few shot samples
many_shot_samples = ""
for essay, score in few_shot_df.sort(by=ATT).select(['essay', ATT]).iter_rows():
    sample = f"score: {score}\nessay text: {essay}\n"
    many_shot_samples += sample

# load md file
user_message = Path(LLM_PROMPT_PATH).read_text()
essay_prompt = (Path(ESSAY_PROMPT_DIR) / f"prompt_{TARGET_PROMPT_ID}.md").read_text()
rubric = list((Path(ESSAY_RUBRIC_DIR) / ATT).glob(f'*_{TARGET_PROMPT_ID}*'))[0].read_text()

for id, essay in tqdm(test_df.sort(by='essay_id').select(['essay_id', 'essay']).iter_rows()):
    message = (
        user_message
            .replace('{prompt}', essay_prompt)
            .replace('{rubric}', rubric)
            .replace('{examples}', many_shot_samples)
            .replace('{minscore}', str(minscore))
            .replace('{maxscore}', str(maxscore))
            .replace("{essay}", essay)
    )

    queries[id] = [
        {'role': 'user', 'content': message}
    ]

1753it [00:00, 7304.29it/s]


In [74]:
import tiktoken


def count_total_tokens(queries, model="gpt-4o-mini"):
    enc = tiktoken.encoding_for_model(model)
    total_tokens = 0

    for qlist in queries.values():
        for msg in qlist:
            total_tokens += len(enc.encode(msg["content"]))

    return total_tokens


total = count_total_tokens(queries)
print(f"総トークン数: {total:,}")

総トークン数: 23,978,317


In [75]:
from tokenrail import BatchExecutor, PerRequestJsonSink, RailClient, ResultsJsonlSink, RollingMetricsMonitor
from tokenrail.executor import batch_items_from_queries

def main():
    client = RailClient.vllm(
        model_id=MODEL,
        family=MODEL_FAMILIY,
        batch_flush_size=256,
        dtype="bfloat16",
        max_model_len=32000,
        gpu_memory_utilization=0.92,
        enable_prefix_caching=True,
        trust_remote_code=True,
        seed=12,
    )

    items = batch_items_from_queries(
        queries,
        model=MODEL,
        max_output_tokens=64,
        enable_thinking=False,
    )

    executor = BatchExecutor(
        client=client,
        sinks=[
            ResultsJsonlSink(JSONL_PATH),
            PerRequestJsonSink(OUTDIR / "requests"),
        ],
        monitor=RollingMetricsMonitor(),
    )

    stats = executor.run(items)
    print(stats.to_dict())

if __name__ == "__main__":
    main()

/home/takumi/Documents/aes-llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 04-21 20:34:06 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'seed': 12, 'max_model_len': 32000, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.92, 'disable_log_stats': True, 'model': 'Qwen/Qwen3.5-9B'}
INFO 04-21 20:34:28 [model.py:549] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 04-21 20:34:28 [model.py:1678] Using max model len 32000
INFO 04-21 20:34:28 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 04-21 20:34:28 [config.py:441] Mamba cache mode is set to 'align' for Qwen3_5ForConditionalGeneration by default when prefix caching is enabled
INFO 04-21 20:34:28 [config.py:461] Warning: Prefix caching in Mamba cache 'align' mode is currently enabled. Its support for Mamba layers is experimental. Please report any issues you may observe.


`Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


INFO 04-21 20:34:30 [config.py:281] Setting attention block size to 528 tokens to ensure that attention page size is >= mamba page size.
INFO 04-21 20:34:30 [config.py:312] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
INFO 04-21 20:34:30 [vllm.py:790] Asynchronous scheduling is enabled.


The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


...

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 21/21 [00:00<00:00, 28.12it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 21/21 [00:00<00:00, 27.09it/s]


(EngineCore pid=33642) INFO 04-21 20:35:32 [gpu_model_runner.py:6046] Graph capturing finished in 2 secs, took 0.30 GiB
(EngineCore pid=33642) INFO 04-21 20:35:32 [gpu_worker.py:597] CUDA graph pool memory: 0.3 GiB (actual), 0.08 GiB (estimated), difference: 0.22 GiB (73.1%).
(EngineCore pid=33642) INFO 04-21 20:35:32 [core.py:283] init engine (profile, create kv cache, warmup model) took 21.45 seconds
(EngineCore pid=33642) INFO 04-21 20:35:32 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-21 20:35:32 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.


(EngineCore pid=33642) /home/takumi/Documents/aes-llm/.venv/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (15) < num_heads (16). This may indicate the inputs were passed in head-first format [B, H, T, ...] Please verify your input tensor format matches the expected shape [B, T, H, ...].
(EngineCore pid=33642)   return fn(*args, **kwargs)
(EngineCore pid=33642) /home/takumi/Documents/aes-llm/.venv/lib/python3.11/site-packages/vllm/model_executor/layers/fla/ops/utils.py:113: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (15) < num_heads (32). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
(EngineCore pid=33642)   return fn(*contiguous_args, **contiguous_kwargs)


[1/1753] id=1 model=Qwen/Qwen3.5-9B status=ok elapsed=00:11:53 eta=347:01:13 finish=07:47:06 in=13831 cached=0 out=11 reasoning=0 total=13842 rpm=1 tpm=13842 cost=$0.000000 payer=- developer_total=$0.000000 openai_total=$0.000000
[2/1753] id=2 model=Qwen/Qwen3.5-9B status=ok elapsed=00:11:53 eta=173:24:41 finish=02:10:33 in=13908 cached=0 out=11 reasoning=0 total=13919 rpm=2 tpm=27761 cost=$0.000000 payer=- developer_total=$0.000000 openai_total=$0.000000
[3/1753] id=3 model=Qwen/Qwen3.5-9B status=ok elapsed=00:11:53 eta=115:32:29 finish=16:18:22 in=13743 cached=0 out=11 reasoning=0 total=13754 rpm=3 tpm=41515 cost=$0.000000 payer=- developer_total=$0.000000 openai_total=$0.000000
[4/1753] id=4 model=Qwen/Qwen3.5-9B status=ok elapsed=00:11:53 eta=86:36:24 finish=11:22:17 in=14136 cached=0 out=11 reasoning=0 total=14147 rpm=4 tpm=55662 cost=$0.000000 payer=- developer_total=$0.000000 openai_total=$0.000000
[5/1753] id=5 model=Qwen/Qwen3.5-9B status=ok elapsed=00:11:53 eta=69:14:44 finis

[rank0]:[W421 20:45:53.774516300 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


# calculate QWK

In [76]:
import json
from json_repair import repair_json

results = []
errors = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            entry = json.loads(line.strip())
        except json.JSONDecodeError:
            continue  # JSONとして読み込めない行はスキップ

        entry_id = entry['id']
        text = entry.get("output_text", "")

        try:
            good_json = repair_json(text)
            score = json.loads(good_json)['score']
            results.append({"essay_id": int(entry_id), "pred_score": score})
        except:
            results.append({"essay_id": int(entry_id), "pred_score": None})
            errors.append(f"ID {entry_id} のスコアが不正な形式です: {text}")


print(f"抽出結果: {len(results)}件、エラー: {len(errors)}件")

抽出結果: 1753件、エラー: 0件


In [77]:
results_df = pl.DataFrame(results)
results_df

essay_id,pred_score
i64,i64
1,4
2,7
3,4
4,7
5,7
…,…
1783,7
1784,4
1785,2


In [78]:
test_df

essay_id,essay,overall
i64,str,i64
1746,"""Computers can take a lot of a …",10
445,"""Dear @CAPS1, In my opinion I t…",8
1647,"""Dear newspaper, Technology is …",6
1143,"""To the @CAPS1, I have recently…",8
1228,"""Hi! I am writing in which comp…",8
…,…,…
983,"""The computer is a good way to …",8
209,"""Dear local newspaper editor, @…",6
949,"""Many People in Society think c…",8


In [79]:
final_df = test_df.join(results_df, on='essay_id', how='left')
final_df

essay_id,essay,overall,pred_score
i64,str,i64,i64
1746,"""Computers can take a lot of a …",10,4
445,"""Dear @CAPS1, In my opinion I t…",8,2
1647,"""Dear newspaper, Technology is …",6,2
1143,"""To the @CAPS1, I have recently…",8,7
1228,"""Hi! I am writing in which comp…",8,4
…,…,…,…
983,"""The computer is a good way to …",8,2
209,"""Dear local newspaper editor, @…",6,2
949,"""Many People in Society think c…",8,7


In [81]:
from sklearn.metrics import cohen_kappa_score

minscore, maxscore = get_min_max_scores(TARGET_PROMPT_ID, ATT)
qwk = cohen_kappa_score(
    final_df["pred_score"].to_numpy(),
    final_df[ATT].to_numpy(),
    weights="quadratic",
    labels=list(range(minscore, maxscore + 1)),
)
print(f"QWK: {qwk:.4f}")

QWK: 0.1987


# Many-shot Performance

| Prompt.| QWK |
|----:|-------:|
| 1   | 0.5109 |
| 2   | 0.6056 |
| 3   | 0.5126 |
| 4   | 0.7858 |
| 5   | 0.7376 |
| 6   | 0.8018 |
| 7   | 0.6457 |
| 8   | 0.6737 |

avg = 0.6592